In [1]:
import pandas as pd 
import requests
import re
import numpy as np
%run -i geocoding_functions.py


In [4]:
df = pd.read_csv("../../data/data_cleaned/biais_cleaned/patients_adresse_id.csv", sep=";",dtype=str)
df_all_info = pd.read_csv("../../data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv", sep=";",dtype=str)


In [ ]:
liste_bruit = df[~pd.isna(df['bruit'])]['pseudo_provisoire'].tolist()
df['adresse_cleaned'] = df['numeros'] + ' ' + df['voirie'] + ' ' + df['elem_adresse']

df_adresse_cleaned = df[['pseudo_provisoire','adresse_cleaned','codepost','nom_commune_postal']]


In [ ]:

df_adresse_cleaned['requete'] = df_adresse_cleaned['adresse_cleaned'] + ' ' + df_adresse_cleaned['codepost'] + ' ' + df_adresse_cleaned['nom_commune_postal']


In [ ]:
# df_geocoded_cleaned = geocode(df_adresse_cleaned)
# df_geocoded_cleaned.to_csv("../../data/data_cleaned/biais_cleaned/patients_adresse_cleaned_geocoded.csv", sep=";")


In [ ]:
df_1_geoc = pd.read_csv("../../data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv",sep=";",dtype={'codepost':str})
df_1_geoc = df_1_geoc[['pseudo_provisoire','adresse','codepost','x', 'y', 'score']]
df_1_geoc = df_1_geoc.rename(columns = {'adresse' : 'adresse_1','codepost' : 'codepost_1','x' : 'x1','y' : 'y1','score':'score_1'})
df_1_geoc['pseudo_provisoire'] = df_1_geoc['pseudo_provisoire'].astype(str)



In [ ]:
df_comp = df_geocoded_cleaned.merge(df_1_geoc, on='pseudo_provisoire')


In [ ]:
df_comp['adresse_1'] = df_comp['adresse_1'].apply(lambda x: separer_alphanumerique(x.split(" ")))

df_comp['diff_score'] = df_comp['score_1'].astype(float) - df_comp['score'].astype(float)
df_comp['same_adresse'] = df_comp['adresse_1'] == df_comp['adresse_cleaned']
df_comp['diff_taille_adresse'] = np.nan
for i in df_comp.index:
    if pd.isna(df_comp.loc[i,'adresse_1']) is False and pd.isna(df_comp.loc[i,'adresse_cleaned']) is False:
        df_comp.at[i,'diff_taille_adresse'] = len(df_comp.loc[i,'adresse_1'].split(' ')) - len(df_comp.loc[i,'adresse_cleaned'].split(' '))


In [ ]:
# df_comp[df_comp['diff_taille_adresse'] < 0]['diff_taille_adresse'].mean()#[['adresse_cleaned','adresse_1','diff_taille_adresse']]
# df_comp.loc[df_comp['diff_taille_adresse'].isnull()][['adresse_cleaned','adresse_1']]

df_comp[df_comp['pseudo_provisoire'].isin(liste_bruit)][['diff_score']].mean()
